In [ ]:
import os
import random
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src.utils import (
    load_image,
    parse_stenosis_xml,
    get_processed_image_paths,
    get_skeleton_path,
    get_bbox_xml_path,
    group_paths_by_serie
)

# ============================================================
# CARGA DE IMÁGENES PREPROCESADAS
# ============================================================

paths = get_processed_image_paths()

print(f"Total processed images found: {len(paths)}")

series = group_paths_by_serie(paths)

print(f"Total series found: {len(series)}")

# ============================================================
# SELECCIÓN ALEATORIA DE SERIES
# ============================================================

SEED = 716
MAX_IMAGES = 50

random.seed(SEED)

selected_series_keys = random.sample(
    list(series.keys()),
    min(MAX_IMAGES, len(series))
)

print(f"Selected series: {len(selected_series_keys)}")

# ============================================================
# PARÁMETROS
# ============================================================

ROI_SIZE = 80

# 99 MONTECARLO + 1 ROI OBLIGATORIA
N_MONTECARLO = 99

half_size = ROI_SIZE // 2

# ============================================================
# DATASET FINAL
# ============================================================

dataset_rows = []

# ============================================================
# LOOP PRINCIPAL
# ============================================================

for serie_key in selected_series_keys:

    serie_paths = series[serie_key]

    # --------------------------------------------------------
    # FRAME CENTRAL
    # --------------------------------------------------------

    path = serie_paths[len(serie_paths) // 2]

    frame_name = os.path.basename(path).replace(
        "_processed.png",
        ""
    )

    print(f"\nProcessing: {frame_name}")

    # --------------------------------------------------------
    # CARGAR IMAGEN PROCESADA
    # --------------------------------------------------------

    proc = load_image(path)

    if proc is None:
        continue

    # --------------------------------------------------------
    # CARGAR SKELETON
    # --------------------------------------------------------

    skeleton_path = get_skeleton_path(path)

    skeleton_img = load_image(skeleton_path)

    if skeleton_img is None:
        continue

    skeleton = skeleton_img > 0

    # --------------------------------------------------------
    # CARGAR XML
    # --------------------------------------------------------

    boxes = parse_stenosis_xml(
        get_bbox_xml_path(path)
    )

    if not boxes:
        continue

    # --------------------------------------------------------
    # EXTRAER PUNTOS DEL SKELETON
    # --------------------------------------------------------

    y_coords, x_coords = np.where(skeleton)

    puntos_disponibles = set(
        zip(x_coords, y_coords)
    )

    puntos_ordenados = []

    # --------------------------------------------------------
    # WALKER SIMPLE
    # --------------------------------------------------------

    if len(puntos_disponibles) > 0:

        actual = list(puntos_disponibles)[0]

        puntos_ordenados.append(actual)

        puntos_disponibles.remove(actual)

        while len(puntos_disponibles) > 0:

            pts_restantes = np.array(
                list(puntos_disponibles)
            )

            distancias = np.sum(
                (pts_restantes - np.array(actual)) ** 2,
                axis=1
            )

            idx_min = np.argmin(distancias)

            actual = tuple(
                pts_restantes[idx_min]
            )

            puntos_ordenados.append(actual)

            puntos_disponibles.remove(actual)

    # --------------------------------------------------------
    # MUESTREO MONTECARLO
    # --------------------------------------------------------

    total_puntos = len(puntos_ordenados)

    montecarlo_boxes = []

    if total_puntos > 0:

        indices_equidistantes = np.linspace(
            0,
            total_puntos - 1,
            N_MONTECARLO,
            dtype=int
        )

        # ====================================================
        # 99 WINDOWS MONTECARLO
        # ====================================================

        for idx in indices_equidistantes:

            pt_x, pt_y = puntos_ordenados[idx]

            mc_xmin = int(pt_x - half_size)
            mc_ymin = int(pt_y - half_size)

            mc_xmax = int(pt_x + half_size)
            mc_ymax = int(pt_y + half_size)

            # ------------------------------------------------
            # AJUSTE A BORDES
            # ------------------------------------------------

            if mc_xmin < 0:

                mc_xmax -= mc_xmin
                mc_xmin = 0

            if mc_ymin < 0:

                mc_ymax -= mc_ymin
                mc_ymin = 0

            if mc_xmax >= proc.shape[1]:

                mc_xmin -= (
                    mc_xmax - proc.shape[1] + 1
                )

                mc_xmax = proc.shape[1] - 1

            if mc_ymax >= proc.shape[0]:

                mc_ymin -= (
                    mc_ymax - proc.shape[0] + 1
                )

                mc_ymax = proc.shape[0] - 1

            montecarlo_boxes.append({

                'xmin': mc_xmin,
                'ymin': mc_ymin,
                'xmax': mc_xmax,
                'ymax': mc_ymax
            })

        # ====================================================
        # +1 ROI OBLIGATORIA CENTRADA EN LA ESTENOSIS
        # ====================================================

        for box in boxes:

            center_x = int(
                (box['xmin'] + box['xmax']) / 2
            )

            center_y = int(
                (box['ymin'] + box['ymax']) / 2
            )

            roi_xmin = center_x - half_size
            roi_ymin = center_y - half_size

            roi_xmax = center_x + half_size
            roi_ymax = center_y + half_size

            # ------------------------------------------------
            # AJUSTE A BORDES
            # ------------------------------------------------

            if roi_xmin < 0:

                roi_xmax -= roi_xmin
                roi_xmin = 0

            if roi_ymin < 0:

                roi_ymax -= roi_ymin
                roi_ymin = 0

            if roi_xmax >= proc.shape[1]:

                roi_xmin -= (
                    roi_xmax - proc.shape[1] + 1
                )

                roi_xmax = proc.shape[1] - 1

            if roi_ymax >= proc.shape[0]:

                roi_ymin -= (
                    roi_ymax - proc.shape[0] + 1
                )

                roi_ymax = proc.shape[0] - 1

            montecarlo_boxes.append({

                'xmin': roi_xmin,
                'ymin': roi_ymin,
                'xmax': roi_xmax,
                'ymax': roi_ymax
            })

    # ========================================================
    # GUARDAR INFO EN DATASET
    # ========================================================

    for idx, mc_box in enumerate(montecarlo_boxes):

        pintar_rojo = False

        for box in boxes:

            inter_xmin = max(
                mc_box['xmin'],
                box['xmin']
            )

            inter_ymin = max(
                mc_box['ymin'],
                box['ymin']
            )

            inter_xmax = min(
                mc_box['xmax'],
                box['xmax']
            )

            inter_ymax = min(
                mc_box['ymax'],
                box['ymax']
            )

            inter_w = max(
                0,
                inter_xmax - inter_xmin
            )

            inter_h = max(
                0,
                inter_ymax - inter_ymin
            )

            area_interseccion = inter_w * inter_h

            area_box_original = (
                (box['xmax'] - box['xmin']) *
                (box['ymax'] - box['ymin'])
            )

            if (
                area_box_original > 0
                and
                area_interseccion >= 0.40 * area_box_original
            ):

                pintar_rojo = True
                break

        # ----------------------------------------------------
        # LABEL FINAL
        # ----------------------------------------------------

        window_label = 1 if pintar_rojo else 0

        # ----------------------------------------------------
        # ID ÚNICO
        # ----------------------------------------------------

        window_id = f"{frame_name}_{idx + 1}"

        # ----------------------------------------------------
        # GUARDAR FILA
        # ----------------------------------------------------

        dataset_rows.append({

            'window_id': window_id,

            'image_path': path,

            'xmin': mc_box['xmin'],
            'ymin': mc_box['ymin'],
            'xmax': mc_box['xmax'],
            'ymax': mc_box['ymax'],

            'label': window_label
        })

    # ========================================================
    # VISUALIZACIÓN
    # ========================================================

    visual_final = cv2.cvtColor(
        proc,
        cv2.COLOR_GRAY2BGR
    )

    # Skeleton azul
    visual_final[skeleton] = [255, 0, 0]

    # --------------------------------------------------------
    # GT ORIGINAL
    # --------------------------------------------------------

    for box in boxes:

        cv2.rectangle(
            visual_final,
            (box['xmin'], box['ymin']),
            (box['xmax'], box['ymax']),
            color=(255, 0, 255),
            thickness=2
        )

    # --------------------------------------------------------
    # WINDOWS
    # --------------------------------------------------------

    for mc_box in montecarlo_boxes:

        pintar_rojo = False

        for box in boxes:

            inter_xmin = max(
                mc_box['xmin'],
                box['xmin']
            )

            inter_ymin = max(
                mc_box['ymin'],
                box['ymin']
            )

            inter_xmax = min(
                mc_box['xmax'],
                box['xmax']
            )

            inter_ymax = min(
                mc_box['ymax'],
                box['ymax']
            )

            inter_w = max(
                0,
                inter_xmax - inter_xmin
            )

            inter_h = max(
                0,
                inter_ymax - inter_ymin
            )

            area_interseccion = inter_w * inter_h

            area_box_original = (
                (box['xmax'] - box['xmin']) *
                (box['ymax'] - box['ymin'])
            )

            if (
                area_box_original > 0
                and
                area_interseccion >= 0.40 * area_box_original
            ):

                pintar_rojo = True
                break

        color_box = (
            (0, 0, 255)
            if pintar_rojo
            else
            (0, 255, 0)
        )

        cv2.rectangle(
            visual_final,
            (mc_box['xmin'], mc_box['ymin']),
            (mc_box['xmax'], mc_box['ymax']),
            color=color_box,
            thickness=1
        )

    # --------------------------------------------------------
    # MOSTRAR
    # --------------------------------------------------------

    plt.figure(figsize=(10, 10))

    plt.imshow(
        cv2.cvtColor(
            visual_final,
            cv2.COLOR_BGR2RGB
        )
    )

    plt.title(frame_name)

    plt.axis('off')

    plt.show()

# ============================================================
# CREAR DATAFRAME
# ============================================================

df = pd.DataFrame(dataset_rows)

print(df.head())

print(f"\nTotal windows: {len(df)}")

# ============================================================
# GUARDAR CSV
# ============================================================

csv_name = "stenosis_windows_dataset.csv"

df.to_csv(
    csv_name,
    index=False
)

print(f"\nCSV guardado en: {csv_name}")